# 5. Conclusion, Recommendations & Future Work

This notebook summarises the key findings from the Metro Interstate Traffic Volume project (UCI Repository ID 492), covering data collected on I-94 westbound between Minneapolis and St. Paul from 2012 to 2018.

## 5.1 Main Findings

### Data Profile
| Metric | Value |
|--------|-------|
| Total hourly records | 48,204 |
| Date range | Oct 2012 – Sep 2018 |
| Records after cleaning | 40,564 |
| Duplicate / missing rows removed | ~7,640 |

### Model Results (Chronological 80/20 Split)
| Model | MAE | RMSE | R² |
|-------|-----|------|----|
| **XGBoost** | 234.07 | 379.52 | **0.9629** |
| Random Forest | 250.92 | 411.27 | 0.9564 |
| Linear Regression | 821.57 | 1042.94 | 0.7198 |

The XGBoost pipeline (300 estimators, LR=0.05, depth=6) achieved the best generalisation with an **R² of 0.963** on unseen chronological test data, meaning it explains **96.3 % of variance** in hourly traffic volume.

### Top Traffic Drivers

1. **Time of day** — The single strongest predictor. Weekday rush hours (7–9 AM and 4–6 PM) account for the sharpest peaks, reaching 6,000–7,000 vehicles/hr. Night hours drop below 500.

2. **Day of week** — Weekday traffic is roughly 1.5× higher than weekend traffic during peak windows. Sunday shows the flattest, lowest-volume profile.

3. **Weather conditions** — Heavy snowfall reduces throughput by up to 32 %. Rain reduces it by 8–18 %. Clear, mild days produce and sustain the highest volumes.

4. **Holiday effect** — On public holidays the typical morning commute peak disappears almost entirely, replaced by a gentler midday curve at ~60–65 % of normal volume.

### Key Statistics
- **Median traffic volume:** ~3,200 vehicles/hr (all hours)
- **Peak average (weekday 8 AM):** ~6,100 vehicles/hr
- **Minimum average (3 AM):** ~380 vehicles/hr
- **Train R² vs Test R²:** 0.969 vs 0.963 → minimal overfitting

## 5.2 Business Recommendations

### For Traffic Management Agencies

| Recommendation | Rationale |
|---|---|
| Deploy adaptive signal timing 7–9 AM & 4–6 PM on weekdays | These windows drive the highest volume and largest safety risk |
| Activate road-gritting crews proactively when snowfall forecast > 2 mm | Model shows snowfall cuts capacity by up to 32 % |
| Relax ramp-metering on Saturday/Sunday mornings | Weekend volumes are 30–40 % lower; over-restriction wastes capacity |
| Issue holiday traffic advisories 1–2 hrs earlier than normal | Holiday peaks shift to midday rather than morning |

### For Route Planning Apps & Navigation

- Integrate the XGBoost pipeline as a **real-time congestion predictor**: given current weather and time, produce a volume estimate within milliseconds.
- Use **hour-range forecasting** (as exposed in the production API) to proactively reroute users before they enter a congested window.

### For Urban Planners

- The data confirms a strong **bi-modal commute pattern** on I-94 — future infrastructure investment should focus on peak-hour throughput rather than average daily traffic.
- Staggered work-hour policies (e.g., flex-start at 10 AM) could reduce morning peak volumes by an estimated 15–20 % based on the diurnal curve analysis.

## 5.3 Limitations & Future Work

### Current Limitations

| Limitation | Impact |
|---|---|
| Dataset is from 2012–2018 — pre-pandemic | Post-2020 work-from-home patterns may shift the diurnal curve significantly |
| Single road segment (I-94 westbound only) | Cannot generalise to other interstates or directions without retraining |
| Weather features are hourly averages, not sub-hourly | Extreme short-duration weather events (flash floods, sudden snowfall) may be under-captured |
| No incident / accident data | Traffic incidents can cause sudden non-weather volume drops that the model cannot predict |
| Live weather API limited to city-level resolution | On-road micro-weather may differ from city observation |

### Recommended Future Improvements

1. **Retrain on post-2020 data** — capture the structural shift in commute patterns caused by hybrid work.

2. **Add incident features** — integrate MnDOT real-time incident feed to allow the model to distinguish weather-driven slowdowns from crash-driven ones.

3. **Sequence models (LSTM / Transformer)** — exploit temporal autocorrelation. Volume at hour *t* is strongly correlated with volume at *t-1* and *t-2*; a recurrent model may further reduce MAE by 10–15 %.

4. **Multi-segment modelling** — extend to all I-94 detector stations between Minneapolis and St. Paul to enable full corridor-level forecasting.

5. **Uncertainty quantification** — replace point estimates with prediction intervals (e.g., XGBoost quantile regression) so that planners can act on worst-case vs. best-case scenarios.

6. **Automated model retraining** — set up a monthly retraining pipeline triggered when model drift is detected on live prediction errors.

## 5.4 Final Note

The project demonstrates that a well-engineered XGBoost pipeline — trained on publicly available weather and calendar features — can predict interstate traffic volume with **96.3 % explained variance** and a mean absolute error of only **234 vehicles/hour** out of a typical peak-hour volume of 6,000+.

The model is now deployed as a production REST API with a live web dashboard that allows planners and commuters to forecast traffic for any hour range over a 3-day window, powered by real-time weather data.

---
*Dataset: Metro Interstate Traffic Volume · UCI ML Repository ID 492 · CC BY 4.0*